<a href="https://www.kaggle.com/code/adityasajgotra/cancer-detection?scriptVersionId=315748585" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tensorflow as tf
from tensorflow import keras
import os

In [ ]:
tf.random.set_seed(42)

In [ ]:
path_part_1 = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1"
path_part_2 = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2"

In [ ]:
plt.figure(figsize=(15,5))
for i in range(0,10):
    plt.subplot(2,5,i + 1)
    if i < 5:
        img = cv2.imread(os.path.join(path_part_1, os.listdir(path_part_1)[i]))
    else:
        img = cv2.imread(os.path.join(path_part_2, os.listdir(path_part_2)[i]))
    plt.imshow(img)


In [ ]:
img1 = cv2.imread(os.path.join(path_part_1, os.listdir(path_part_1)[0]))
img2 = cv2.imread(os.path.join(path_part_1, os.listdir(path_part_1)[1]))
print(img1.shape)
print(img2.shape)

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_metadata.csv")

In [ ]:
df.info()

In [ ]:
df.sample(5)

In [ ]:
df["dx"].value_counts()/len(df)

In [ ]:
x = df.iloc[:,:-1]
y = df.iloc[:,2]

In [ ]:
df.info()

In [ ]:
df.isna().sum()/len(df)


In [ ]:
df.dropna(inplace=True,axis=0)

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

train_idx, test_idx = next(sgkf.split(df, df["dx"], groups=df["lesion_id"]))


In [ ]:
train = df.iloc[train_idx]
test = df.iloc[test_idx]

In [ ]:
train.info()

In [ ]:
train["dx"].value_counts()/len(train) #ratio of each class

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer,make_column_selector
from sklearn.preprocessing import OneHotEncoder, StandardScaler,OrdinalEncoder

num_pip = Pipeline([
    ('scaler',StandardScaler())
])
cat_pip = Pipeline([
    ("encode",OneHotEncoder(sparse_output=False))
])

label_pip = Pipeline([
    ("label",OrdinalEncoder())
])

full_pipeline = ColumnTransformer([
    ("num",num_pip,['age']),
    ("cat",cat_pip,["sex","localization","dx"])

],remainder="passthrough")

In [ ]:
imgs = tf.data.Dataset.list_files("/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/*/*.jpg")

In [ ]:
for i in imgs:
    print(tf.strings.split(tf.strings.split(i,os.path.sep)[-1],".")[0])
    break
#tf.strings.split(tf.strings.split(i,os.path.sep)[-1],".")[0] extracts the id from the image file name

In [ ]:
train.sample(5)

In [ ]:
final_train = full_pipeline.fit_transform(train)

In [ ]:
final_test = full_pipeline.transform(test)

In [ ]:
full_pipeline.get_feature_names_out()

In [ ]:
train = pd.DataFrame(
    final_train,columns = full_pipeline.get_feature_names_out()
)
test = pd.DataFrame(
    final_test,columns = full_pipeline.get_feature_names_out()
)

In [ ]:
train.sample(5)

In [ ]:
train.columns

In [ ]:
arr = [0]*19  + ["","","","",""]
arr

In [ ]:
def make_csv(train,test):
    img_file1 = os.listdir("/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1")
    img_file1 = set(map(lambda x : x.split(".")[0],img_file1))
    path1 = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_1/"
    path2 = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000/HAM10000_images_part_2/"
    train["path"] = np.where(train["remainder__image_id"].isin(img_file1), path1+train["remainder__image_id"]+".jpg",path2+train["remainder__image_id"]+".jpg")
    test["path"] = np.where(test["remainder__image_id"].isin(img_file1), path1+test["remainder__image_id"]+".jpg",path2+test["remainder__image_id"]+".jpg")
    test.to_csv("test.csv",index=False)
    train.to_csv("train.csv",index=False)
    
    print("csv successfully executed")

In [ ]:
make_csv(train,test)

In [ ]:
train = pd.read_csv("/kaggle/working/train.csv")
train.sample(5)

In [ ]:
test = pd.read_csv("/kaggle/working/test.csv")
test.columns == train.columns

In [ ]:
train.columns

In [ ]:
def csv_parser(filepath):
    raw_file = tf.io.read_file(filepath)
    splitted_raw_file = tf.strings.split(raw_file,sep="\n")
    return tf.data.Dataset.from_tensor_slices(splitted_raw_file)

In [ ]:
def map_label(infoline):
    decoded = tf.io.decode_csv(infoline,record_defaults = [[0.0]]*26 + [[""]]*4)
    img_raw = tf.io.read_file(decoded[-1])
    img_decoded = tf.io.decode_jpeg(img_raw,channels=3)
    img_decoded = tf.image.resize(img_decoded,[224,224])/255.0
    label = tf.cast(decoded[-11:-4],tf.float32)
    # return (img_decoded,decoded[:-5]), tf.reshape(label,[1])
    return img_decoded, label
    

In [ ]:
raw = csv_parser("/kaggle/working/train.csv")
raw = raw.skip(1)
raw = raw.take(len(raw)-1)

In [ ]:
raw_test = csv_parser("/kaggle/working/test.csv")
raw_test = raw_test.skip(1)
raw_test = raw_test.take(len(raw_test)-1)

In [ ]:
train_ds = raw.map(map_label)
test_ds = raw_test.map(map_label)

In [ ]:
count = len(train_ds)
count



In [ ]:
len(test_ds)

In [ ]:
train_ds = train_ds.shuffle(
    buffer_size=count, 
    seed=42, 
    reshuffle_each_iteration=False
)
len(train_ds)

In [ ]:
test_ds = test_ds.shuffle(
    buffer_size=len(test_ds), 
    seed=42, 
    reshuffle_each_iteration=False
)

In [ ]:

val_ds = train_ds.skip(int(count * 0.8)).batch(64).prefetch(tf.data.AUTOTUNE)
train_ds = train_ds.take(int(count * 0.8)).batch(64).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.batch(64).prefetch(tf.data.AUTOTUNE)

In [ ]:
len(train_ds)

In [ ]:
import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import Dense,Input,RandomFlip,RandomRotation,GlobalMaxPooling2D,Concatenate,Dropout,GlobalAveragePooling2D

In [ ]:
model_pre = tf.keras.applications.MobileNetV2(
    input_shape=(224,224,3),
    weights='imagenet',
    include_top = False
)

In [ ]:
model_pre.trainable = True

In [ ]:
from tensorflow.keras import layers, regularizers


img_input = Input(shape=(224,224,3))
x = layers.RandomFlip("horizontal")(img_input)
x = layers.RandomRotation(0.1)(x)
model_augment = model_pre(x)
model_out = GlobalMaxPooling2D()(model_augment)
model_out = layers.BatchNormalization()(model_out)

# concat = Concatenate()([model_out,data_inp])

# dense2 = Dense(256,activation="relu")(model_out)
dense2 = Dense(128,activation="relu")(model_out)
# dense3 = Dense(64,activation="relu")(dense2)
dense2 = Dropout(0.4)(dense2)
# dense4 = Dense(32,activation="relu")(dense3)
# dense5 = Dense(16,activation="relu")(dense4)
# dense5 = Dropout(0.3)(dense5)
dense7 = Dense(7,activation="softmax")(dense2)

main_model = Model(inputs=[img_input],outputs=[dense7])

In [ ]:
# from sklearn.utils import class_weight
# import numpy as np

# weights = class_weight.compute_class_weight(
#     class_weight='balanced',
#     classes=np.unique(train['label__dx']),
#     y=train['label__dx']
# )

# class_weights_dict = dict(enumerate(weights))

# print(class_weights_dict)


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early = tf.keras.callbacks.EarlyStopping(
    monitor='val_auc',
    min_delta=0.00001,
    patience=10,
    verbose=1,
    mode='auto',
    baseline=None,
    restore_best_weights=True,
    start_from_epoch=0
)

In [ ]:
from tensorflow.keras import layers, regularizers

main_model.compile(
    optimizer="adam",
    loss=tf.keras.losses.CategoricalFocalCrossentropy(gamma=2), 
    metrics=[tf.keras.metrics.AUC(multi_label=True,num_labels = 7,name="auc")]
)


In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

lr = ReduceLROnPlateau(
    monitor="val_auc",
    factor=0.1,
    patience=5,
    verbose=1,
    mode="auto",
    min_delta=0.00001,
    cooldown=0,
    min_lr=1e-6
)

In [ ]:
main_model.fit(
    train_ds,
    epochs=300,
    validation_data=val_ds, 
    callbacks=[early,lr]
    # class_weight=class_weights_dict
)


In [ ]:
import keras_tuner as kt
import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras import layers
from tensorflow.keras.layers import Dense,Input,RandomFlip,RandomRotation,GlobalMaxPooling2D,Concatenate,Dropout
def build_model(hp):
    img_input = Input(shape=(224,224,3))
    x = layers.RandomFlip("horizontal")(img_input)
    x = layers.RandomRotation(0.1)(x)
    model_augment = model_pre(x)
    model_out = GlobalMaxPooling2D()(model_augment)
    model_out = layers.BatchNormalization()(model_out)
    dense2 = Dense( units=hp.Int("units_1", 128, 512, step=64),activation=hp.Choice("act_1",["relu","elu","selu","tanh"]))(model_out)
    # dense3 = Dense(units=hp.Int("units_2",64,256,step=64),activation=hp.Choice("act_2",["relu","selu","elu","tanh"]))(dense2)
    dense2 = Dropout(rate=hp.Float("drop_1",0.1,0.5,step=0.1))(dense2)
    # dense4 = Dense(units=hp.Int("units_3",32,128,step=32),activation=hp.Choice("act_3",["relu","selu","elu","tanh"]))(dense3)
    # dense5 = Dense(units=hp.Int("units_4",16,48,step=16),activation=hp.Choice("act_4",["relu","selu","elu","tanh"]))(dense4)
    # dense5 = Dropout(rate=hp.Float("drop_2",0.1,0.5,step=0.1))(dense5)
    dense7 = Dense(7,activation="softmax")(dense2)

    main_model_tuned = Model(inputs=[img_input],outputs=[dense7])



    main_model_tuned.compile(
        optimizer="adam",
        loss=tf.keras.losses.CategoricalFocalCrossentropy(gamma=2), 
        metrics=[tf.keras.metrics.AUC(multi_label=True,num_labels = 7,name="auc")]
    )

    return main_model_tuned

In [ ]:
for x, y in train_ds:
    print("Batch Input Shape:", x.shape)
    print("Batch Labels:", y.numpy()) 
    import numpy as np
    print("Unique labels in this batch:", np.unique(y.numpy()))
    break


In [ ]:
import numpy as np

print(f"{'Batch':<8} | {'Unique Preds':<15} | {'Classes Predicted'}")
print("-" * 50)

# Loop through all batches in your test dataset
for i, (x, y) in enumerate(test_ds):
    preds = tuned_imported.predict(x, verbose=0)
    
    # 1. Count how many different 'probability arrays' are in this batch
    # (Rounded to 5 decimals to ignore tiny floating point math differences)
    unique_count = len(np.unique(np.round(preds, 5), axis=0))
    
    # 2. Get the index of the class the model actually chose (0-6)
    predicted_classes = np.argmax(preds, axis=1)
    unique_classes = np.unique(predicted_classes)
    
    # 3. Get the index of the actual correct labels for comparison
    actual_classes = np.unique(np.argmax(y, axis=1))
    
    print(f"Batch {i:<2} | {unique_count:<15} | Predicted: {unique_classes} vs Actual: {actual_classes}")

print("-" * 50)
print("Finished checking test_ds.")


In [ ]:
tuner = kt.RandomSearch(
    build_model,
    objective="val_auc",
    max_trials=10,        
    directory='skin_cancer_tuning',
    project_name='random_search_run'
)


In [ ]:
tuner.search(
    train_ds, 
    validation_data=val_ds,  
    callbacks=[early, lr],
    epochs = 15
)

In [ ]:
best_model = tuner.get_best_models(num_models = 1)
main_model_tuned = best_model[0]

In [ ]:
main_model_tuned.fit(train_ds,
    initial_epoch = 15,
    epochs=300,
    validation_data=val_ds, 
    callbacks=[early,lr]
)

In [ ]:
pred = imported_model.predict(test_ds)

In [ ]:
main_model.evaluate(test_ds)

[0.05475369468331337, 0.9751693606376648]

In [ ]:
main_model_tuned.evaluate(test_ds)

In [ ]:
pred.shape

In [ ]:
#baseline model
imported_model = tf.keras.models.load_model("/kaggle/input/models/adityasajgotra/ham-model/keras/default/1/model (1).keras")

In [ ]:
#tuned model
tuned_imported = tf.keras.models.load_model("/kaggle/input/models/adityasajgotra/tuned-ham/keras/default/1/model_tuned.keras")

In [ ]:
tuned_imported.evaluate(train_ds)

In [ ]:
tuned_imported.evaluate(val_ds)

In [ ]:
tuned_imported.evaluate(test_ds)

In [ ]:
pred = tuned_imported.predict(test_ds)

In [ ]:
imported_model.evaluate(val_ds)

In [ ]:
imported_model.evaluate(train_ds)

In [ ]:
imported_model.evaluate(test_ds)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# 1. Get the true labels (ensure they match the order of your 'pred' variable)
# This unbatches the dataset to get all labels as a single array
y_true = np.concatenate([y for x, y in test_ds], axis=0)

# 2. Process your 'pred' (get the index of the highest probability)
y_pred_classes = np.argmax(pred, axis=1)

# 3. Handle y_true if it's one-hot encoded
if len(y_true.shape) > 1:
    y_true_classes = np.argmax(y_true, axis=1)
else:
    y_true_classes = y_true

# 4. Define HAM10000 labels
class_names = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

# 5. Plot
cm = confusion_matrix(y_true_classes, y_pred_classes)
plt.figure(figsize=(10, 8))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap='Blues', values_format='d')
plt.title('HAM10000 Test Confusion Matrix')
plt.xticks(rotation=45)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# --- STEP 1: PREPARE DATA ---
# 1. Get true labels from the dataset (unbatching them)
y_true_raw = np.concatenate([y for x, y in test_ds], axis=0)

# 2. Handle both one-hot and integer labels for y_true
if len(y_true_raw.shape) > 1:
    y_true_indices = np.argmax(y_true_raw, axis=1)
else:
    y_true_indices = y_true_raw

# --- STEP 2: APPLY CUSTOM THRESHOLD ---
# 'pred' is your array from main_model.predict(test_ds)
# Index 4 is 'mel' (Melanoma) in the HAM10000 dataset
MEL_INDEX = 4
SAFETY_THRESHOLD = 0.15  # Lowered from 0.50 to catch more cancers

y_pred_adjusted = []

for p in pred:
    # If the probability of Melanoma is above our safety bar, prioritize it
    if p[MEL_INDEX] >= SAFETY_THRESHOLD:
        y_pred_adjusted.append(MEL_INDEX)
    else:
        # Otherwise, pick the class with the highest probability
        y_pred_adjusted.append(np.argmax(p))

y_pred_adjusted = np.array(y_pred_adjusted)

# --- STEP 3: PLOT THE NEW CONFUSION MATRIX ---
class_names = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
cm = confusion_matrix(y_true_indices, y_pred_adjusted)

plt.figure(figsize=(12, 9))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap='Blues', values_format='d')

plt.title(f'HAM10000 Matrix (Melanoma Threshold: {SAFETY_THRESHOLD})')
plt.xticks(rotation=45)
plt.show()

# --- STEP 4: PRINT SENSITIVITY CHECK ---
# Calculate how many Melanomas were caught vs total
total_mel = np.sum(y_true_indices == MEL_INDEX)
caught_mel = cm[MEL_INDEX, MEL_INDEX]
missed_mel = total_mel - caught_mel

print(f"Total Melanomas in Test Set: {total_mel}")
print(f"Melanomas Correctly Caught: {caught_mel} ({ (caught_mel/total_mel)*100 :.2f}%)")
print(f"Melanomas Missed: {missed_mel}")


In [ ]:
main_model.export("model_tf")

In [ ]:
main_model.save("model.keras")

In [ ]:
main_model_tuned.export("model_tuned_tf")

In [ ]:
main_model_tuned.save("model_tuned.keras")